# Síntesis e Integración de Técnicas de Procesamiento del Lenguaje Natural

**Materiales desarrollados por Matías Barreto, 2026**  
**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**

- **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
- **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

## Pregunta del recorrido

¿Cómo pasa un texto de una cadena de caracteres a una representación que permite compararlo, describirlo y clasificarlo?

El cuaderno conserva el recorrido completo: cada celda de código muestra un paso. El texto indica qué operación se realiza, qué resultado hay que mirar y qué decisión implica.

## Al terminar vas a poder

1. Comparar formas de normalizar y tokenizar textos en español.
2. Calcular distancias de edición y proponer coincidencias difusas.
3. Construir y leer matrices BoW, TF-IDF y n-gramas.
4. Contrastar stemming, lematización, categorías gramaticales, dependencias y entidades.
5. Comparar similitud de vectores densos con similitud de vectores TF-IDF.
6. Entrenar un clasificador de polaridad y auditar sus errores.

## Microglosario

| Término | Qué hace en este cuaderno |
|---|---|
| **Token** | Unidad obtenida al segmentar un texto: palabra, número o signo. |
| **Normalización** | Transformación de formas superficiales, como mayúsculas o tildes, según una regla declarada. |
| **Distancia de Levenshtein** | Número mínimo de inserciones, eliminaciones o sustituciones para convertir una cadena en otra. |
| **Bolsa de palabras** | Tabla donde cada documento es una fila y cada término una columna de frecuencias. |
| **TF-IDF** | Ponderación que destaca términos frecuentes en un documento y poco frecuentes en el corpus. |
| **N-grama** | Secuencia de \(n\) elementos contiguos, como `buenos aires`. |
| **Stemming** | Recorte de terminaciones mediante reglas; no verifica si el resultado es una palabra de diccionario. |
| **Lematización** | Reducción de una forma flexionada a una forma canónica, como `analizaron` → `analizar`. |
| **POS** | Categoría gramatical propuesta para cada token. |
| **Vector denso** | Representación numérica de dimensión fija, distinta de una matriz dispersa de conteos. |

## 1. Configuración del entorno y reproducibilidad
**Operación.** Importar las herramientas, fijar una semilla, descargar recursos de NLTK y cargar el modelo de spaCy.
**Por qué importa.** El resultado depende de versiones, recursos y particiones aleatorias. Registrar esas condiciones permite repetir el procedimiento y comparar cambios.
**Antes de ejecutar.** Usá el entorno virtual del curso y confirmá que el mensaje final muestre las versiones cargadas.
**Nota sobre recursos.** Este cuaderno utiliza  (modelo grande de spaCy, ~500 MB, con 500.000 vectores de 300 dimensiones). Si ejecutás este material en una máquina con memoria RAM ajustada (menos de 8 GB) o conexión lenta, podés sustituirlo por  (modelo mediano, ~40 MB, también con vectores densos).


In [ ]:
# 1. Importamos herramientas para expresiones regulares y normalización de caracteres.
import re
import unidecode

# 2. Importamos herramientas para cálculos numéricos y manejo de tablas.
import numpy as np
import pandas as pd

# 3. Importamos herramientas para crear gráficos y comparar distribuciones.
import matplotlib.pyplot as plt
import seaborn as sns

# 4. Importamos recursos de PLN: tokenización, distancia de edición, stemming y palabras vacías.
import nltk
from nltk.metrics import edit_distance
from nltk.stem import SnowballStemmer

# 5. Importamos búsqueda aproximada: 'fuzz' define puntajes y 'process' compara listas.
from thefuzz import fuzz, process

# 6. Importamos análisis lingüístico y visualización de etiquetas y dependencias.
import spacy
from spacy import displacy

# 7. Importamos scikit-learn para vectorizar textos, dividir datos, entrenar y evaluar modelos.
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 8. La semilla hace repetibles las operaciones aleatorias; podés cambiar 42 por otro entero
# para obtener otra partición reproducible y comparar cuánto varían los resultados.
SEMILLA = 42
np.random.seed(SEMILLA)

# 9. Guardamos los recursos de NLTK dentro del entorno virtual para no dispersar archivos en el sistema.
import sys
from pathlib import Path
ruta_nltk = Path(sys.prefix) / "nltk_data"
nltk.download("punkt", download_dir=str(ruta_nltk), quiet=True)
nltk.download("punkt_tab", download_dir=str(ruta_nltk), quiet=True)
nltk.download("stopwords", download_dir=str(ruta_nltk), quiet=True)

# 10. El modelo grande aporta vectores para la comparación semántica posterior.
# Podés cambiar el nombre por otro modelo de español instalado, pero sus vectores pueden variar.
nlp = spacy.load("es_core_news_lg")

# 11. Verificamos versiones, modelo y semilla para poder interpretar y repetir el análisis.
print(f"Pandas: {pd.__version__} | spaCy: {spacy.__version__}")
print(f"Modelo spaCy: {nlp.meta['name']}-{nlp.meta['version']}")
print(f"Dimensiones de sus vectores: {nlp.vocab.vectors.shape}")
print(f"Semilla fijada: {SEMILLA}")

**Leé el resultado.** Confirmá tres datos: versión de spaCy, nombre del modelo y dimensiones de sus vectores. Esos datos sitúan los resultados posteriores; no garantizan por sí solos que el análisis sea válido.

## 2. Carga y exploración del corpus de noticias

**Operación.** Cargar una muestra de noticias de sitios `.ar` y revisar sus columnas y primeros registros.

**Decisión.** Trabajar con textos periodísticos locales permite observar vocabulario, instituciones y ruido propios de ese material. No convierte la muestra en representación de toda la prensa argentina.

In [ ]:
# Cambiá esta ruta relativa si el archivo Parquet está en otra carpeta o tiene otro nombre.
ruta_noticias = "datos/cc-news-es/ar_muestra_5k.parquet"

# Cargamos el archivo Parquet en una tabla de pandas.
corpus_noticias = pd.read_parquet(ruta_noticias)

# Revisamos tamaño, columnas y primeras filas antes de transformar el texto.
print(f"Total de artículos cargados: {len(corpus_noticias):,}")
print(f"Columnas disponibles: {corpus_noticias.columns.tolist()}")
print()
print("Primeros registros:")
corpus_noticias.head(3)

**Leé la tabla.** `country` identifica el dominio geográfico, `id` identifica el documento y `text` contiene el cuerpo de la nota. Antes de modelar, conviene saber qué columnas existen y cómo está almacenado el texto.

In [ ]:
# 1. Medimos la longitud de cada artículo en caracteres.
longitudes_caracteres = corpus_noticias["text"].str.len()

# 2. Resumimos la distribución para ver textos cortos, medianos y extremos.
print("Longitud de los textos, en caracteres:")
print(longitudes_caracteres.describe().apply(lambda valor: f"{valor:,.1f}"))

# 3. Leemos un fragmento real para observar puntuación, nombres e instituciones.
# iloc[0] muestra la primera fila; cambiá el índice si querés inspeccionar otra noticia.
print()
print("Fragmento del primer artículo:")
print("-" * 80)
print(corpus_noticias["text"].iloc[0][:350] + "...")
print("-" * 80)

**Leé el resultado.** Compará mínimo, mediana y máximo. La dispersión importa: una misma técnica puede comportarse distinto en una nota breve y en un artículo muy extenso.

**Probá.** Elegí otro índice en `iloc[...]` y observá si aparecen siglas, citas, nombres propios o fragmentos difíciles de segmentar.

## 3. Del carácter al token

**Pregunta.** ¿Qué unidades obtiene cada forma de segmentar la misma oración?

Usaremos una oración testigo para comparar tres procedimientos: separar por espacios, aplicar una expresión regular y usar el tokenizador de NLTK.

In [ ]:
# 1. Definimos una oración que contiene coma, tilde, guion y punto.
oracion_testigo = "Usualmente, existe una relación costo-beneficio entre las distintas técnicas."

# 2. Primer procedimiento: separamos únicamente por espacios.
tokens_primitivos = oracion_testigo.split(" ")

# 3. Mostramos los elementos y su cantidad.
print("Tokenización con str.split(' '):")
print(tokens_primitivos)
print(f"Cantidad de elementos: {len(tokens_primitivos)}")

**Leé la lista.**

- `Usualmente,` conserva la coma.
- `técnicas.` conserva el punto.
- `costo-beneficio` queda como una sola cadena.

Esas diferencias importan: para una computadora, `técnicas.` y `técnicas` son cadenas distintas.

### Normalización de caracteres

**Operación.** Pasar cada elemento a minúsculas y quitar diacríticos con `unidecode`.

**Límite.** Esta transformación puede ayudar a unificar variantes gráficas, pero también puede borrar distinciones como `público` y `publicó`.

In [ ]:
# 1. Creamos una lista vacía para guardar las formas transformadas.
tokens_normalizados = []

# 2. Recorremos cada elemento y aplicamos dos cambios: minúsculas y eliminación de tildes.
for palabra in tokens_primitivos:
    palabra_limpia = unidecode.unidecode(palabra.lower().strip())
    tokens_normalizados.append(palabra_limpia)

# 3. Comparamos el resultado con la lista anterior.
print("Tokens después de lower() y unidecode:")
print(tokens_normalizados)

**Compará.** `relación` pasa a `relacion`, pero la puntuación sigue adherida. La normalización de caracteres no resuelve por sí sola la segmentación.

### Expresiones regulares

**Operación.** Usar `re.findall()` para extraer secuencias de caracteres de palabra.

`r"\w+"` busca una o más letras, números o guiones bajos. El prefijo `r` indica una cadena cruda, recomendable para escribir patrones.

In [ ]:
# 1. Pasamos la oración a minúsculas.
oracion_en_minusculas = oracion_testigo.lower()

# 2. El patrón r"\w+" extrae letras, números y guiones bajos; ajustalo si necesitás otras unidades.
tokens_regex = re.findall(r"\w+", oracion_en_minusculas)

# 3. Mostramos qué unidades conserva este patrón.
print("Tokens extraídos con re.findall:")
print(tokens_regex)
print(f"Cantidad de tokens: {len(tokens_regex)}")

**Compará con `split()`.**

- Desaparecen la coma y el punto.
- `costo-beneficio` se separa en `costo` y `beneficio`.
- No se conservan los signos como tokens propios.

**Límite.** Este patrón es útil, pero no es una tokenización universal: trata números, guiones y abreviaturas según una regla simple.

In [ ]:
# NLTK separa palabras y signos; 'language' elige los recursos de segmentación en español.
# Cambialo solo si analizás otra lengua compatible con los recursos instalados.
tokens_nltk = nltk.word_tokenize(oracion_testigo, language="spanish")

# Observamos si separa signos y cómo trata el término con guion.
print("Tokens generados por nltk.word_tokenize:")
print(tokens_nltk)
print(f"Cantidad de tokens: {len(tokens_nltk)}")

**Compará las tres listas.** NLTK separa coma y punto, pero conserva `costo-beneficio` como una unidad. Ningún método es correcto en abstracto: la elección depende de si los signos, los guiones o las palabras compuestas importan para la tarea.

### Distancia de edición

**Operación.** Calcular cuántos cambios mínimos separan dos cadenas.

La distancia de Levenshtein cuenta inserciones, eliminaciones y sustituciones. Una distancia baja sugiere similitud superficial; no demuestra que ambas cadenas tengan el mismo significado.

In [ ]:
# 1. Definimos pares para observar distintos tipos de diferencia.
pares_de_palabras = [
    ("pero", "perro"),
    ("pero", "pierdo"),
    ("nueve", "mueve"),
    ("totalmente", "diferentes"),
    ("comunicacion", "comunicación"),
    ("algoritmo", "algoritmo"),
]

# 2. Calculamos la distancia de cada par y mostramos el resultado.
print("Distancia de Levenshtein:")
print("-" * 55)

for palabra_a, palabra_b in pares_de_palabras:
    distancia = edit_distance(palabra_a, palabra_b)
    print(f"{palabra_a:15} vs {palabra_b:15} -> {distancia}")

**Leé cada valor.**

- `pero` / `perro`: una inserción.
- `pero` / `pierdo`: dos cambios.
- `comunicacion` / `comunicación`: una sustitución vinculada al diacrítico.
- Dos cadenas idénticas tienen distancia 0.

**Probá.** Agregá un par de tu corpus y anticipá la distancia antes de ejecutar.

### Búsqueda difusa

**Operación.** Comparar una consulta escrita con errores contra una lista de candidatos y ordenar los resultados por puntaje.

La función siguiente no decide automáticamente cuál candidato es correcto. Devuelve una lista ordenada para que podamos inspeccionar el criterio.

In [ ]:
def buscar_coincidencias_difusas(termino_busqueda, lista_candidatos):
    """Ordena candidatos según su similitud con el término buscado."""
    termino_limpio = termino_busqueda.strip()

    # 'scorer' define cómo se calcula la similitud; token_set_ratio compara conjuntos de palabras.
    # Podés usar fuzz.ratio si querés que importe más la secuencia exacta de caracteres.
    resultados = process.extractBests(
        termino_limpio,
        lista_candidatos,
        scorer=fuzz.token_set_ratio,
    )
    return resultados


# 1. Definimos una consulta con un error intencional y una lista de referencia.
termino_ingresado = "Pebro"
candidatos_padron = ["María", "Fabio", "Paula", "Natu", "Pedro", "Miguel"]

# 2. Buscamos coincidencias y recorremos el resultado para leerlo línea por línea.
coincidencias = buscar_coincidencias_difusas(termino_ingresado, candidatos_padron)

print(f"Búsqueda ingresada: {termino_ingresado}")
print("Candidatos ordenados por similitud:")

for nombre, puntaje in coincidencias:
    print(f"{nombre:10} -> {puntaje}%")

**Leé el orden.** El puntaje propone candidatos; no justifica aceptarlos sin un umbral y una revisión. En nombres de personas, instituciones o direcciones, un falso positivo puede ser más grave que no encontrar una coincidencia automática.

## 4. Representación matricial: conteo, TF-IDF y n-gramas

**Pregunta.** ¿Cómo se transforma una colección de textos en una tabla numérica?

Primero usaremos tres documentos breves para ver el mecanismo. Después aplicaremos n-gramas al corpus de noticias.

In [ ]:
# 1. Creamos una colección pequeña para inspeccionar cada columna.
documentos_muestra = [
    "El procesamiento del lenguaje natural analiza textos y discursos.",
    "El aprendizaje automático entrena modelos para clasificar textos.",
    "Los modelos de lenguaje procesan textos en lenguaje natural con gramática y sintaxis.",
]

# 2. Convertimos los documentos en una matriz de frecuencias.
vectorizador_conteo = CountVectorizer()
matriz_conteo = vectorizador_conteo.fit_transform(documentos_muestra)

# 3. Recuperamos los nombres de las columnas: el vocabulario.
vocabulario_conteo = vectorizador_conteo.get_feature_names_out()

# 4. Inspeccionamos dimensiones, tamaño del vocabulario y tipo de objeto.
print(f"Dimensiones (documentos x términos): {matriz_conteo.shape}")
print(f"Términos únicos: {len(vocabulario_conteo)}")
print(f"Tipo de estructura: {type(matriz_conteo)}")

**Leé la salida.** La matriz tiene una fila por documento y una columna por término. `scikit-learn` devuelve una matriz dispersa: almacena principalmente las posiciones con valores distintos de cero. Para leerla, la convertiremos en una tabla.

In [ ]:
# 1. Convertimos la matriz dispersa en una tabla visible.
df_conteo = pd.DataFrame(
    data=matriz_conteo.toarray(),
    columns=vocabulario_conteo,
    index=[f"Documento_{i + 1}" for i in range(len(documentos_muestra))],
)

# 2. Mostramos la tabla completa porque esta colección es pequeña.
df_conteo

**Leé la tabla.** Cada fila es el vector de un documento. Un valor indica cuántas veces aparece el término de esa columna. Esta representación conserva frecuencias, pero no conserva el orden completo de la oración.

### TF-IDF

**Operación.** Reemplazar el conteo bruto por una ponderación que reduce la influencia relativa de términos compartidos por muchos documentos.

La idea general es:

$$
\operatorname{tfidf}(t, d) = \operatorname{tf}(t, d) \times \operatorname{idf}(t)
$$

`TfidfVectorizer` aplica además sus propias decisiones de suavizado y normalización. Por eso leeremos los valores producidos por esta implementación, no una fórmula aislada.

In [ ]:
# 1. Creamos un vectorizador TF-IDF y lo ajustamos a los mismos documentos.
vectorizador_tfidf = TfidfVectorizer()
matriz_tfidf = vectorizador_tfidf.fit_transform(documentos_muestra)

# 2. Recuperamos su vocabulario y convertimos la matriz en una tabla.
vocabulario_tfidf = vectorizador_tfidf.get_feature_names_out()

df_tfidf = pd.DataFrame(
    data=matriz_tfidf.toarray(),
    columns=vocabulario_tfidf,
    index=[f"Documento_{i + 1}" for i in range(len(documentos_muestra))],
)

# cmap elige la paleta de colores; podés cambiar "Blues" por otra para mejorar el contraste.
# Solo cambia la presentación de la tabla, no los pesos TF-IDF.
df_tfidf.style.background_gradient(cmap="Blues")

**Leé el color.** Las celdas más oscuras corresponden a pesos más altos en esta tabla. Comparalas con términos compartidos, como `textos`: aparecer varias veces no implica automáticamente recibir el mayor peso.

### Similitud coseno

**Operación.** Comparar dos vectores mediante el ángulo que forman.

$$
\operatorname{sim}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}
$$

En esta tabla los pesos son no negativos, por lo que los valores obtenidos quedan entre 0 y 1.

In [ ]:
def calcular_similitud_coseno(vector_a, vector_b):
    """Calcula la similitud coseno entre dos vectores."""
    producto_escalar = np.dot(vector_a, vector_b)
    norma_a = np.linalg.norm(vector_a)
    norma_b = np.linalg.norm(vector_b)

    if norma_a == 0 or norma_b == 0:
        return 0.0

    return producto_escalar / (norma_a * norma_b)


# 1. Extraemos cada documento como un vector.
vectores_tfidf = matriz_tfidf.toarray()
vec_doc1 = vectores_tfidf[0]
vec_doc2 = vectores_tfidf[1]
vec_doc3 = vectores_tfidf[2]

# 2. Comparamos el documento 1 con los otros dos.
sim_1_2 = calcular_similitud_coseno(vec_doc1, vec_doc2)
sim_1_3 = calcular_similitud_coseno(vec_doc1, vec_doc3)

# 3. Mostramos los valores calculados, sin redactarlos manualmente.
print(f"Similitud coseno entre Doc 1 y Doc 2: {sim_1_2:.4f}")
print(f"Similitud coseno entre Doc 1 y Doc 3: {sim_1_3:.4f}")

**Interpretá el número impreso.** La comparación indica cercanía dentro de esta representación TF-IDF. No demuestra que un documento “signifique” lo mismo que otro ni que la similitud sea estable si cambiamos el corpus o el preprocesamiento.

### N-gramas sobre el corpus real

**Operación.** Contar secuencias de dos y tres palabras en los primeros 1000 caracteres de cada noticia.

Recortar la longitud estandariza el cómputo, pero también es una decisión: deja fuera el resto de cada artículo.

In [ ]:
# 1. Recortamos cada artículo a sus primeros 1000 caracteres.
# Cambiá 1000 para incluir un fragmento más corto o más largo; el recorte reduce el costo,
# pero deja fuera el resto de cada nota.
textos_noticias = corpus_noticias["text"].str.slice(0, 1000)

# 2. Elegimos bigramas y trigramas; (2, 2) deja solo pares y (1, 2) agrega palabras sueltas.
# min_df=20 exige que cada secuencia aparezca en 20 documentos; bajalo para incluir rarezas
# o subilo para reducir ruido y tamaño del vocabulario.
vectorizador_ngramas = CountVectorizer(ngram_range=(2, 3), min_df=20)
matriz_ngramas = vectorizador_ngramas.fit_transform(textos_noticias)

# 3. Sumamos cada columna para obtener la frecuencia total de cada n-grama.
frecuencias_ngramas = np.asarray(matriz_ngramas.sum(axis=0)).flatten()
vocabulario_ng = vectorizador_ngramas.get_feature_names_out()

# 4. Ordenamos los resultados de mayor a menor frecuencia.
df_top_ngramas = pd.DataFrame(
    {"frecuencia": frecuencias_ngramas},
    index=vocabulario_ng,
).sort_values(by="frecuencia", ascending=False)

print(f"N-gramas únicos identificados: {len(vocabulario_ng):,}")
print("Primeros 5 n-gramas:")
print(df_top_ngramas.head(5))

**Siguiente paso.** El conteo anterior muestra frecuencia, pero una tabla breve no alcanza para comparar la distribución. Por eso graficamos los 20 n-gramas más frecuentes.

In [ ]:
# head(20) define cuántas expresiones se muestran; cambiá 20 para ampliar o reducir la lista.
top_20_ngramas = df_top_ngramas.head(20)["frecuencia"].sort_values(ascending=True)

# Los parámetros gráficos ajustan la presentación, pero no modifican las frecuencias.
plt.figure(figsize=(10, 6))
top_20_ngramas.plot(kind="barh", color="#2b5c8f", edgecolor="black")
plt.title("20 n-gramas más frecuentes, sin filtrar palabras vacías", fontsize=13, pad=12)
plt.xlabel("Frecuencia absoluta", fontsize=11)
plt.ylabel("N-grama", fontsize=11)
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

**Leé el gráfico.** Si predominan secuencias como `de la` o `en el`, la frecuencia está mostrando conectores muy comunes. Eso no significa que “no tengan significado”: significa que, para describir temas, esta representación todavía no separa función gramatical y contenido temático.

### Filtrado de palabras vacías

**Operación.** Volver a contar n-gramas, pero indicando una lista de palabras vacías en español.

**Decisión.** Filtrar puede hacer visibles expresiones temáticas. También puede eliminar pistas relevantes para estudiar negación, modalidad, estilo o autoría.

In [ ]:
# 1. Cargamos y revisamos la lista de palabras vacías en español.
# Cambiar 'spanish' solo corresponde si se analiza otra lengua con recursos instalados.
palabras_vacias_espanol = nltk.corpus.stopwords.words("spanish")

print(f"Cantidad de palabras vacías: {len(palabras_vacias_espanol)}")
print(f"Primeros elementos: {palabras_vacias_espanol[:10]}")

# 2. Excluimos palabras vacías y contamos bigramas y trigramas.
# ngram_range=(2, 3) puede cambiarse a (2, 2) para conservar solo pares.
# min_df=10 mantiene secuencias presentes en 10 documentos; bajalo para recuperar rarezas,
# o subilo para restringir el vocabulario a expresiones más repetidas.
# stop_words usa la lista anterior; modificarla cambia qué palabras pueden formar cada n-grama.
vectorizador_ng_filtrado = CountVectorizer(
    ngram_range=(2, 3),
    min_df=10,
    stop_words=palabras_vacias_espanol,
)
matriz_ng_filtrado = vectorizador_ng_filtrado.fit_transform(textos_noticias)

# 3. Calculamos frecuencias y ordenamos el nuevo vocabulario.
frecuencias_filtradas = np.asarray(matriz_ng_filtrado.sum(axis=0)).flatten()
vocab_filtrado = vectorizador_ng_filtrado.get_feature_names_out()

df_ng_filtrado = pd.DataFrame(
    {"frecuencia": frecuencias_filtradas},
    index=vocab_filtrado,
).sort_values(by="frecuencia", ascending=False)

df_ng_filtrado.head(10)

**Siguiente paso.** Graficamos la versión filtrada para compararla visualmente con el gráfico anterior.

In [ ]:
# head(20) limita la cantidad de n-gramas visibles; cambiá 20 para mostrar más o menos.
top_20_filtrados = df_ng_filtrado.head(20)["frecuencia"].sort_values(ascending=True)

# El tamaño y el color afectan solo la lectura del gráfico, no los conteos calculados.
plt.figure(figsize=(10, 6))
top_20_filtrados.plot(kind="barh", color="#1b7837", edgecolor="black")
plt.title("20 n-gramas más frecuentes, después de filtrar palabras vacías", fontsize=13, pad=12)
plt.xlabel("Frecuencia absoluta", fontsize=11)
plt.ylabel("N-grama", fontsize=11)
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

**Compará ambos gráficos.** Registrá qué expresiones aparecen después del filtro y cuáles siguen siendo ambiguas. Una expresión frecuente puede señalar un tema, un lugar, una persona o una fórmula periodística; la frecuencia sola no distingue esos casos.

In [ ]:
# Agregá o quitá frases exactas para excluirlas del ranking, sin borrar sus palabras por separado.
frases_a_excluir = ["si bien", "cada vez", "primera vez", "provincia buenos", "provincia buenos aires"]

# 'errors="ignore"' permite continuar si una frase no aparece en otro corpus o configuración.
df_ngramas_sin_frases = df_ng_filtrado.drop(
    index=frases_a_excluir,
    errors="ignore",
)

# Cambiá 20 para mostrar más o menos expresiones; solo modifica el tamaño del gráfico.
top_20_sin_frases = df_ngramas_sin_frases.head(20)["frecuencia"]
top_20_sin_frases = top_20_sin_frases.sort_values(ascending=True)

# El tamaño y el color cambian la presentación, no el resultado del conteo.
plt.figure(figsize=(10, 6))
top_20_sin_frases.plot(kind="barh", color="#6c1b78", edgecolor="black")
plt.title("20 n-gramas más frecuentes, sin las frases seleccionadas")
plt.xlabel("Frecuencia absoluta")
plt.ylabel("N-grama")
plt.grid(axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

### Stemming con NLTK

**Operación.** Aplicar reglas de recorte a distintas formas de una palabra y observar qué raíz produce el algoritmo.

El stemming no consulta el sentido de la oración. Por eso puede agrupar formas útiles, pero también unir palabras que no queremos tratar como equivalentes.

In [ ]:
# 1. Creamos un stemmer configurado para español.
stemmer_es = SnowballStemmer("spanish")

# 2. Elegimos formas para observar recortes verbales y un posible sobrecorte.
palabras_ejemplo = ["mirar", "miraré", "mirando", "codear", "codeando", "hacer", "hachas"]

# 3. Aplicamos el stemmer a cada palabra y mostramos la pareja original-resultado.
print("Stemming con Snowball:")
print("-" * 50)

for palabra in palabras_ejemplo:
    raiz = stemmer_es.stem(palabra)
    print(f"{palabra:10} -> {raiz}")

### Qué mirar en el stemming

- Puede reducir muchas formas a una misma representación.
- Es rápido y no necesita un modelo lingüístico grande.
- Puede producir **sobrecorte**: `hacer` y `hachas` pueden acercarse a una raíz similar aunque no convenga unirlas.

La próxima sección compara este recorte con la lematización de spaCy.

## 5. Análisis lingüístico con spaCy

**Pregunta.** ¿Qué información agrega un análisis de tokens, lemas, categorías, dependencias y entidades?

**Operación.** Procesar una oración con `nlp(...)` y recorrer el objeto `Doc` que devuelve spaCy.

Estas anotaciones son propuestas de un modelo entrenado. Sirven para analizar, pero deben revisarse cuando el texto es ambiguo o pertenece a un dominio distinto del entrenamiento.

In [ ]:
# 1. Definimos una oración con verbo principal, subordinación y signos.
texto_complejo = (
    "El Doctor afirmó que actuando de maneras probabilísticas el cerebro "
    "aprende conceptos discretos. ¡Qué fenómeno misterioso!"
)

# 2. Procesamos el texto con el modelo cargado al inicio.
doc_spacy = nlp(texto_complejo)

# 3. Inspeccionamos el tipo de objeto y los tokens reconocidos.
print(f"Tipo de objeto: {type(doc_spacy)}")
print(f"Cantidad de tokens: {len(doc_spacy)}")
print()
print("Tokens individuales:")
print([token.text for token in doc_spacy])

### Lematización frente a stemming

**Operación.** Para cada token alfabético, guardar el texto original, el lema propuesto por spaCy y el stem producido por NLTK.

In [ ]:
# 1. Creamos una lista de registros para construir una tabla comparativa.
comparativa_morfologica = []

# 2. Recorremos los tokens y conservamos solo los formados por letras.
for token in doc_spacy:
    if token.is_alpha:
        stem_nltk = stemmer_es.stem(token.text)
        comparativa_morfologica.append({
            "Token": token.text,
            "Lema (spaCy)": token.lemma_,
            "Stem (NLTK)": stem_nltk,
        })

# 3. Convertimos los registros en una tabla.
df_comparativa = pd.DataFrame(comparativa_morfologica)
df_comparativa

**Leé tres filas.**

- `afirmó`: el lema busca el infinitivo `afirmar`; el stem recorta la forma.
- `actuando`: el lema busca `actuar`; el stem produce una raíz abreviada.
- `probabilísticas`: el lema conserva una forma legible de diccionario.

La lematización no es siempre superior para toda tarea, pero suele ser más interpretable cuando una persona necesita leer el vocabulario resultante.

### Oraciones y categorías gramaticales

**Operación.** Primero listamos las oraciones detectadas. Después armamos una tabla con la categoría de cada token.

In [ ]:
# 1. Recorremos las oraciones detectadas por el modelo.
print("Oraciones detectadas:")

# start=1 numera desde uno; cambialo a 0 si necesitás índices que empiecen en cero.
# Solo modifica la numeración que se imprime, no la segmentación de spaCy.
for indice, oracion in enumerate(doc_spacy.sents, start=1):
    print(f"{indice}. {oracion.text.strip()}")

`token.pos_` muestra una categoría general y `token.tag_` una etiqueta más específica. `spacy.explain()` ayuda a traducir esa etiqueta a una descripción breve.

In [ ]:
# 1. Creamos un registro por token con su categoría y su descripción.
registros_pos = []

for token in doc_spacy:
    registros_pos.append({
        "Token": token.text,
        "Categoría (POS)": token.pos_,
        "Etiqueta fina (Tag)": token.tag_,
        "Descripción": spacy.explain(token.pos_),
    })

# 2. Mostramos las primeras filas para leer el mecanismo sin perdernos en la tabla.
pd.DataFrame(registros_pos).head(8)

### Dependencias sintácticas

**Operación.** Tomar la primera oración y dibujar su árbol de dependencias.

Cada arco propone una relación entre una palabra y su núcleo, como sujeto, objeto o modificador.

In [ ]:
# 1. Seleccionamos la primera oración detectada.
primera_oracion = list(doc_spacy.sents)[0]

# 2. style="dep" dibuja relaciones sintácticas; distance separa los arcos para facilitar su lectura.
# Aumentá distance si el gráfico se superpone; color y bg solo modifican su apariencia.
displacy.render(
    primera_oracion.as_doc(),
    style="dep",
    jupyter=True,
    options={"distance": 110, "color": "#060606", "bg": "#f8f9fa"},
)

**Leé el gráfico en tres pasos.**

1. Encontrá el verbo principal.
2. Buscá qué token aparece como sujeto.
3. Seguí la cláusula introducida por `que`.

Esta estructura permite formular preguntas que una bolsa de palabras no representa directamente. Aun así, el gráfico es una hipótesis automática y puede fallar en textos ambiguos o mal segmentados.

### Extracción de relaciones sintácticas con código

**Operación.** Recorrer los tokens de la oración para extraer programáticamente el verbo principal () y su sujeto nominal (), sin depender de la inspección visual.


In [ ]:
# 1. Recorremos los tokens buscando el núcleo del predicado y su sujeto nominal.
verbo_nucleo = None
sujeto_nominal = None

for token in primera_oracion:
    if token.dep_ == "ROOT":
        verbo_nucleo = token
    elif token.dep_ == "nsubj":
        sujeto_nominal = token

# 2. Mostramos los elementos estructurales aislados con su categoría y lema.
print("Elementos extraídos del árbol de dependencias:")
print(f"  * Verbo principal (ROOT): {verbo_nucleo.text} (lema: '{verbo_nucleo.lemma_}')")
print(f"  * Sujeto nominal (nsubj): {sujeto_nominal.text} (categoría: {sujeto_nominal.pos_})")


**Leé la salida.** El análisis sintáctico permite aislar quién ejecuta la acción () y cuál es el núcleo de la afirmación (), habilitando tareas de extracción de información estructurada a gran escala.


### Entidades nombradas

**Operación.** Procesar una oración con personas, lugares e instituciones, y listar las entidades detectadas junto con su etiqueta.

In [ ]:
# 1. Definimos una oración con entidades que queremos revisar.
texto_entidades = (
    "Jorge Luis Borges nació en Buenos Aires y dirigió la Biblioteca Nacional. "
    "Años más tarde, la Universidad de Buenos Aires reconoció su legado literario."
)

# 2. Procesamos el texto y visualizamos las entidades.
doc_entidades = nlp(texto_entidades)

# style="ent" resalta entidades; jupyter=True muestra el resultado dentro del cuaderno.
displacy.render(doc_entidades, style="ent", jupyter=True)

# 3. Listamos cada entidad para poder auditar texto, etiqueta y explicación.
print()
print("Entidades extraídas:")

for entidad in doc_entidades.ents:
    explicacion = spacy.explain(entidad.label_)
    print(f"{entidad.text:30} -> {entidad.label_:6} ({explicacion})")

**Auditá la salida.** Comprobá tanto el texto detectado como su etiqueta. Una institución puede aparecer como organización, lugar u otra categoría según el modelo y el contexto. Antes de usar estas anotaciones en una investigación, conviene revisar una muestra y registrar errores.

### Vectores densos y similitud

**Operación.** Convertir tres oraciones en documentos de spaCy, comparar sus vectores y verificar el resultado con la función de similitud coseno escrita antes.

Un valor alto indica proximidad en el espacio del modelo. No demuestra por sí solo que las oraciones tengan el mismo sentido.

In [ ]:
# 1. Procesamos tres oraciones de comparación.
doc_a = nlp("La curiosidad por nuevas ideas impulsa a formular teorías científicas.")
doc_b = nlp("El deseo de conocimiento motiva la creación de modelos de aprendizaje.")
doc_c = nlp("El partido de fútbol terminó sin goles en una tarde lluviosa de domingo.")

# 2. Calculamos la similitud que ofrece spaCy.
sim_a_b = doc_a.similarity(doc_b)
sim_a_c = doc_a.similarity(doc_c)

# 3. Mostramos la dimensión del vector y las dos comparaciones.
print(f"Dimensión del vector de A: {doc_a.vector.shape}")
print(f"Similitud spaCy entre A y B: {sim_a_b:.4f}")
print(f"Similitud spaCy entre A y C: {sim_a_c:.4f}")

# 4. Repetimos la comparación A-B con la función manual.
sim_manual_a_b = calcular_similitud_coseno(doc_a.vector, doc_b.vector)
print(f"Similitud coseno manual entre A y B: {sim_manual_a_b:.4f}")

**Leé los números impresos.** Compará A-B con A-C y verificá si la similitud manual coincide con la de spaCy. Si A y B se acercan aunque compartan pocas palabras idénticas, la proximidad proviene de los vectores del modelo, no de una coincidencia literal.

## 6. Caso aplicado: clasificación de polaridad

**Pregunta.** ¿Cómo entrenamos un clasificador sin evaluar los mismos ejemplos con los que aprendió?

**Operación.** Cargar reseñas etiquetadas, revisar el balance de clases y separar entrenamiento y prueba.

In [ ]:
# Cambiá esta ruta relativa si el archivo CSV está en otra carpeta o tiene otro nombre.
ruta_polaridad = "datos/opiniones_polaridad.csv"
df_polaridad = pd.read_csv(ruta_polaridad)

# Revisamos cantidad, distribución de clases y ejemplos.
print(f"Reseñas cargadas: {len(df_polaridad)}")
print()
print("Distribución de etiquetas:")
print(df_polaridad["sentimiento"].value_counts())
print()
print("Ejemplos por clase:")
df_polaridad.groupby("sentimiento").head(2)

### Separar entrenamiento y prueba

**Operación.** Destinar 75 % de las reseñas a entrenar y 25 % a evaluar.

`stratify` mantiene la proporción de clases en ambas particiones. `random_state=SEMILLA` hace repetible esta división.

In [ ]:
# 1. Separamos los textos de sus etiquetas.
X_textos = df_polaridad["texto"]
y_etiquetas = df_polaridad["sentimiento"]

# 2. Creamos particiones; test_size reserva 25 % para evaluar y 75 % para entrenar.
# Podés cambiar 0.25, por ejemplo a 0.20, si necesitás más datos de entrenamiento.
# random_state usa la semilla fijada: cambiarla crea otra partición, repetible con ese nuevo valor.
# stratify mantiene la proporción de clases; quitá este parámetro solo si buscás otra estrategia.
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
    X_textos,
    y_etiquetas,
    test_size=0.25,
    random_state=SEMILLA,
    stratify=y_etiquetas,
)

# 3. Comprobamos el tamaño y el balance de cada partición.
print(f"Entrenamiento: {len(X_entrenamiento)}")
print(f"  Positivas: {(y_entrenamiento == 'Positivo').sum()}")
print(f"  Negativas: {(y_entrenamiento == 'Negativo').sum()}")
print(f"Prueba: {len(X_prueba)}")
print(f"  Positivas: {(y_prueba == 'Positivo').sum()}")
print(f"  Negativas: {(y_prueba == 'Negativo').sum()}")

### Construir el pipeline

**Operación.** Encadenar dos pasos:

1. `TfidfVectorizer(min_df=2)` convierte textos en vectores TF-IDF y deja fuera términos que aparecen una sola vez.
2. `MultinomialNB()` estima, a partir del entrenamiento, qué clase es más probable para un documento.

El pipeline se ajusta solo con `X_entrenamiento` e `y_entrenamiento`.

In [ ]:
# 1. Encadenamos la vectorización y el clasificador en un solo pipeline.
# min_df=2 conserva términos presentes en al menos dos reseñas; bajalo a 1 para incluir
# términos únicos o subilo para reducir el vocabulario y descartar expresiones raras.
clasificador_polaridad = make_pipeline(
    TfidfVectorizer(min_df=2),
    MultinomialNB(),
)

# 2. Ajustamos el pipeline únicamente con los datos de entrenamiento.
clasificador_polaridad.fit(X_entrenamiento, y_entrenamiento)

# 3. Mostramos los pasos para verificar qué quedó configurado.
print("Pipeline ajustado:")
print(clasificador_polaridad)

### Evaluar con datos no usados para entrenar

**Operación.** Predecir las etiquetas de `X_prueba` y compararlas con `y_prueba`.

`accuracy` resume aciertos globales. El reporte por clase muestra si ese resultado se sostiene para cada etiqueta.

In [ ]:
# 1. Generamos una predicción para cada reseña de prueba.
predicciones_prueba = clasificador_polaridad.predict(X_prueba)

# 2. Comparamos predicciones y etiquetas reales.
exactitud_global = accuracy_score(y_prueba, predicciones_prueba)

# 3. Mostramos el resumen global y el detalle por clase.
print(f"Accuracy en prueba: {exactitud_global:.3f}")
print()
print("Reporte por clase:")
# digits=3 muestra tres decimales; cambialo si necesitás más o menos precisión visual.
# Este ajuste cambia el formato impreso, no las métricas calculadas.
print(classification_report(y_prueba, predicciones_prueba, digits=3))

**Leé el reporte.** No te detengas en el número global. Compará precisión y recall de cada clase, y después usá la matriz de confusión para ver qué tipos de error ocurren.

In [ ]:
# 1. El orden de esta lista fija la orientación de filas y columnas en la matriz.
# Cambialo si necesitás presentar primero otra clase; los conteos no cambian.
etiquetas_ordenadas = ["Positivo", "Negativo"]

# 2. Calculamos la matriz de confusión.
matriz_conf = confusion_matrix(
    y_prueba,
    predicciones_prueba,
    labels=etiquetas_ordenadas,
)

# 3. La visualizamos: filas = etiqueta real; columnas = predicción.
# Los colores, el tamaño y las etiquetas ajustan la presentación, no las predicciones.
plt.figure(figsize=(7, 5))
sns.heatmap(
    matriz_conf,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=etiquetas_ordenadas,
    yticklabels=etiquetas_ordenadas,
    cbar=False,
)
plt.title("Matriz de confusión del clasificador de polaridad", fontsize=12, pad=12)
plt.xlabel("Etiqueta predicha", fontsize=11)
plt.ylabel("Etiqueta real", fontsize=11)
plt.tight_layout()
plt.show()

**Leé la matriz.**

- La diagonal muestra aciertos.
- Fuera de la diagonal aparecen confusiones entre clases.
- Un buen resultado promedio no reemplaza la lectura de ejemplos mal clasificados.

**Probá.** Identificá qué celda corresponde a reseñas negativas predichas como positivas y cuál a la confusión inversa.

### Inspección de términos en el clasificador

**Operación.** Extraer del pipeline los términos del vocabulario con mayor probabilidad logarítmica para cada clase (), conectando la matriz TF-IDF con la decisión del modelo bayesiano.


In [ ]:
# 1. Recuperamos el vectorizador y el estimador desde el pipeline entrenado.
vectorizador_ajustado = clasificador_polaridad.named_steps["tfidfvectorizer"]
modelo_nb = clasificador_polaridad.named_steps["multinomialnb"]

nombres_terminos = vectorizador_ajustado.get_feature_names_out()
clases = modelo_nb.classes_

# 2. Obtenemos los 8 términos con mayor peso condicional para cada polo.
print("Términos más asociados a cada clase por Naive Bayes:")
print("-" * 55)

for indice_clase, nombre_clase in enumerate(clases):
    pesos = modelo_nb.feature_log_prob_[indice_clase]
    top_indices = np.argsort(pesos)[-8:][::-1]
    top_palabras = [nombres_terminos[i] for i in top_indices]
    print(f"[{nombre_clase.upper()}]: {", ".join(top_palabras)}")


**Leé los términos.** Observá qué vocablos dominan cada clase. Esta inspección permite auditar si el clasificador aprende patrones temáticos válidos o si se apoya en sesgos circunstanciales del corpus.


### Probar oraciones nuevas

**Operación.** Definir una función que reciba un texto, prediga su clase y muestre la probabilidad asignada a cada clase.

La probabilidad expresa la confianza del modelo según lo aprendido. No expresa una verdad lingüística independiente del corpus.

In [ ]:
def predecir_polaridad_con_probabilidad(texto_a_evaluar):
    """Predice la clase de un texto y muestra la probabilidad de cada clase."""
    clase_predicha = clasificador_polaridad.predict([texto_a_evaluar])[0]
    probabilidades = clasificador_polaridad.predict_proba([texto_a_evaluar])[0]
    clases_modelo = clasificador_polaridad.classes_

    print(f"Texto: {texto_a_evaluar}")
    print(f"Clase predicha: {clase_predicha}")
    print("Probabilidades:")

    for clase, probabilidad in zip(clases_modelo, probabilidades):
        print(f"  {clase:10} -> {probabilidad:.2%}")

    print("-" * 60)


# 1. Probamos una reseña claramente negativa.
predecir_polaridad_con_probabilidad(
    "El servicio fue desastroso, la comida estaba fría y la atención fue muy mala."
)

# 2. Probamos una reseña claramente positiva.
predecir_polaridad_con_probabilidad(
    "Una experiencia magnífica, las instalaciones impecables y el personal sumamente atento."
)

**Siguiente prueba.** Las dos oraciones anteriores usan léxico muy marcado. Ahora observaremos qué ocurre cuando la polaridad depende de una negación.

## 7. Límites, pruebas y práctica

**Pregunta.** ¿Qué errores sistemáticos puede producir una representación basada en palabras?

Tres casos para observar:

1. **Negación:** `no es bueno` contiene `bueno`, pero invierte su valoración.
2. **Ironía:** palabras positivas pueden usarse con sentido crítico.
3. **Orden:** `el perro mordió al hombre` y `el hombre mordió al perro` pueden producir vectores muy parecidos.

In [ ]:
# 1. Definimos una reseña cuya interpretación depende de la negación.
frase_negacion = "No fue una buena estadía, el servicio no fue agradable y no volvería jamás."

# 2. La evaluamos con la misma función usada para las oraciones anteriores.
predecir_polaridad_con_probabilidad(frase_negacion)

**Interpretá el caso.** Si el modelo acierta, registrá qué palabras pudieron ayudarlo. Si falla o muestra poca diferencia entre clases, relacioná el resultado con los límites de una representación que no modela completamente la composición de la oración.

### Ejercitación guiada

1. Reemplazá la tercera oración por una oración ambigua o con negación.
2. Ejecutá la predicción.
3. Leé la clase asignada y explicá qué palabras pueden estar influyendo.
4. Conservá las comprobaciones técnicas, pero no trates una predicción automática como una respuesta garantizada.

In [ ]:
# 1. Definí tres textos de prueba.
mis_textos_evaluacion = [
    "Pésima experiencia, todo sucio y desagradable.",
    "Hermoso lugar, súper recomendable y cordial.",
    "No fue una buena experiencia, aunque el lugar era amplio.",
]

# 2. Obtené una predicción para cada texto.
mis_predicciones = clasificador_polaridad.predict(mis_textos_evaluacion)

# 3. Mostrá cada texto junto con su predicción.
print("Resultados:")

for texto, prediccion in zip(mis_textos_evaluacion, mis_predicciones):
    print(f"{prediccion:10} <- {texto}")

# 4. Comprobá condiciones técnicas del ejercicio.
assert len(mis_predicciones) == 3, "Debés evaluar exactamente 3 oraciones."
assert set(mis_predicciones).issubset({"Positivo", "Negativo"})

print()
print("Comprobación técnica superada: hay tres predicciones con etiquetas válidas.")
print("Ahora interpretá la tercera oración: ¿la predicción coincide con tu lectura?")

### Práctica autónoma

Reemplazá las dos oraciones por textos de tu ámbito: noticias, comentarios, documentos o reseñas culturales. Antes de ejecutar, escribí en una nota qué clase esperás y qué palabras creés que inclinarán el resultado.

In [ ]:
# 1. Reemplazá estas oraciones por ejemplos propios.
mi_oracion_propia_1 = "La atención fue rápida y resolvieron mi consulta con gran amabilidad."
mi_oracion_propia_2 = "Se demoraron dos horas y la respuesta fue completamente inútil."

# 2. Compará tu expectativa con la predicción y las probabilidades.
predecir_polaridad_con_probabilidad(mi_oracion_propia_1)
predecir_polaridad_con_probabilidad(mi_oracion_propia_2)

## 8. Síntesis del recorrido

```text
Texto
  -> normalización y expresiones regulares
  -> tokenización
  -> distancia de edición y búsqueda difusa
  -> BoW, TF-IDF y n-gramas
  -> stemming, lemas, categorías, dependencias y entidades
  -> vectores densos
  -> clasificación, matriz de confusión y análisis de errores
```

### Decisiones para recordar

1. **Normalizar es una decisión:** unifica variantes, pero puede borrar distinciones.
2. **Filtrar palabras vacías cambia la pregunta:** puede revelar temas, pero también ocultar negación, estilo o modalidad.
3. **Stem y lema no hacen lo mismo:** uno recorta por reglas; el otro propone una forma canónica.
4. **Una métrica alta no cierra el análisis:** hay que leer errores, ejemplos y límites de la representación.

### Referencias

- Bird, S., Klein, E. y Loper, E. (2009). *Natural Language Processing with Python*. O’Reilly.
- Honnibal, M., Montani, I., Van Landeghem, S. y Boyd, A. (2020). *spaCy: Industrial-strength Natural Language Processing in Python*.
- Jurafsky, D. y Martin, J. H. (2024). *Speech and Language Processing* (3.ª ed., borrador).
- Manning, C. D., Raghavan, P. y Schütze, H. (2008). *Introduction to Information Retrieval*. Cambridge University Press.